### Session 9 | Part 1

> In Part 1, you will load a CSV file with Spark, inspect it, and register it as a SQL view.

#### 1. Goal

You will:

- start a Spark session
- load a CSV file with an explicit schema
- inspect schema, rows, columns, and row count
- register a temporary SQL view
- run first Spark SQL checks
- connect the pattern to the final project Spark task

In [ ]:
!pip install pyspark==4.1.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079457 sha256=da108870c2c952befc455bf8404227ce7d5fe68cf1c8bbc1920555df308a7905
  Stored in directory: /root/.cache/pip/wheels/e6/9c/35/b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 4.1.2 which is incompatible.


#### 3. Basics you should know

- Spark can infer CSV schemas, but an explicit schema is safer for important analytics.
- `header=True` tells Spark that the first CSV row contains column names.
- `TimestampType` stores date and time values.
- `createOrReplaceTempView` gives a DataFrame a SQL table name for the current Spark session.
- This session uses service logs. In the final project, the same ideas apply to market data.

Checkpoint question:

Why is an explicit schema useful when loading a CSV?

<details>
<summary>Show answer</summary>

It prevents Spark from guessing important data types incorrectly. This matters for numeric calculations, timestamps, and grouped analytics.

</details>

#### 4. Start Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Session09Part01")
    .master("local[*]")
    .getOrCreate()
)

#### 5. Define the CSV schema

In [2]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

Checkpoint question:

Which columns must be numeric for later calculations?

<details>
<summary>Show answer</summary>

`request_count`, `error_count`, `latency_ms`, `bytes_in`, and `bytes_out` must be numeric because later calculations use division, sums, averages, and rankings.

</details>

#### 6. Load the CSV

In [3]:
#### 6. Load the CSV
events_path = "service_events.csv"

In [4]:
events_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(events_path)
)

#### 7. Inspect the DataFrame

In [5]:
events_df.printSchema()
events_df.show(5, truncate=False)

print("Rows:", events_df.count())
print("Columns:", events_df.columns)

root
 |-- event_id: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- request_count: integer (nullable = true)
 |-- error_count: integer (nullable = true)
 |-- latency_ms: double (nullable = true)
 |-- bytes_in: double (nullable = true)
 |-- bytes_out: double (nullable = true)

+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|event_id|service        |region  |event_time         |request_count|error_count|latency_ms|bytes_in|bytes_out|
+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|1       |auth           |eu-west |2026-05-04 00:00:00|1240         |8          |82.4      |1.45E7  |3.82E7   |
|2       |payments       |eu-west |2026-05-04 00:00:00|860          |21         |146.2     |1.92E7  |4.41E7   |
|3       |search         |us-east |2026-05-04 01:00:00|211

Minimum expected idea:

```txt
Rows: 24
Columns include service, region, event_time, request_count, error_count, latency_ms, bytes_in, bytes_out
event_time is timestamp
```

Checkpoint question:

Which command proves that Spark has loaded all rows?

<details>
<summary>Show answer</summary>

`events_df.count()` proves how many rows Spark loaded.

</details>

#### 8. Register a SQL view

In [6]:
events_df.createOrReplaceTempView("service_events")

Test the view:

In [7]:
spark.sql("""
    SELECT service, region, event_time, request_count
    FROM service_events
    ORDER BY event_time
    LIMIT 10
""").show(truncate=False)

''' Expected output:
+---------------+--------+-------------------+-------------+
|service        |region  |event_time         |request_count|
+---------------+--------+-------------------+-------------+
|auth           |eu-west |2026-05-04 00:00:00|1240         |
|payments       |eu-west |2026-05-04 00:00:00|860          |
|search         |us-east |2026-05-04 01:00:00|2110         |
|recommendations|us-east |2026-05-04 01:00:00|1680         |
|auth           |ap-south|2026-05-04 02:00:00|980          |
|payments       |ap-south|2026-05-04 02:00:00|740          |
|search         |eu-west |2026-05-04 03:00:00|2320         |
|recommendations|eu-west |2026-05-04 03:00:00|1760         |
|auth           |us-east |2026-05-04 09:00:00|1480         |
|payments       |us-east |2026-05-04 09:00:00|1120         |
+---------------+--------+-------------------+-------------+

'''

+---------------+--------+-------------------+-------------+
|service        |region  |event_time         |request_count|
+---------------+--------+-------------------+-------------+
|auth           |eu-west |2026-05-04 00:00:00|1240         |
|payments       |eu-west |2026-05-04 00:00:00|860          |
|search         |us-east |2026-05-04 01:00:00|2110         |
|recommendations|us-east |2026-05-04 01:00:00|1680         |
|auth           |ap-south|2026-05-04 02:00:00|980          |
|payments       |ap-south|2026-05-04 02:00:00|740          |
|search         |eu-west |2026-05-04 03:00:00|2320         |
|recommendations|eu-west |2026-05-04 03:00:00|1760         |
|auth           |us-east |2026-05-04 09:00:00|1480         |
|payments       |us-east |2026-05-04 09:00:00|1120         |
+---------------+--------+-------------------+-------------+



' Expected output:\n+---------------+--------+-------------------+-------------+\n|service        |region  |event_time         |request_count|\n+---------------+--------+-------------------+-------------+\n|auth           |eu-west |2026-05-04 00:00:00|1240         |\n|payments       |eu-west |2026-05-04 00:00:00|860          |\n|search         |us-east |2026-05-04 01:00:00|2110         |\n|recommendations|us-east |2026-05-04 01:00:00|1680         |\n|auth           |ap-south|2026-05-04 02:00:00|980          |\n|payments       |ap-south|2026-05-04 02:00:00|740          |\n|search         |eu-west |2026-05-04 03:00:00|2320         |\n|recommendations|eu-west |2026-05-04 03:00:00|1760         |\n|auth           |us-east |2026-05-04 09:00:00|1480         |\n|payments       |us-east |2026-05-04 09:00:00|1120         |\n+---------------+--------+-------------------+-------------+\n\n'

Checkpoint question:

Does `service_events` become a permanent database table?

<details>
<summary>Show answer</summary>

No. It is a temporary view for the current Spark session. It disappears when the Spark session stops.

</details>

#### 9. Exercise 1: Loading and checking service events

Your notebook should:

1. Start Spark.
2. Define the explicit schema.
3. Load `service_events.csv`.
4. Print the schema, row count, and column names.
5. Register a temporary view named `service_events`.
6. Run SQL queries that answer:
   - How many rows are in the view?
   - Which services appear in the dataset?
   - How many rows does each region have?
7. Stop Spark at the end.

In [8]:
from pyspark.sql import SparkSession

# 1. Start Spark.
spark = (
    SparkSession.builder
    .appName("Session09Exercise01")
    .master("local[*]")
    .getOrCreate()
)

# TODO: define schema = 2. Define the explicit schema.
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

# TODO: load CSV = 3. Load `service_events.csv`.
events_path = "service_events.csv"
events_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(events_path)
)

# TODO: inspect data = 4. Print the schema, row count, and column names.
events_df.printSchema()
events_df.show(5, truncate=False)

print("Rows:", events_df.count())
print("Columns:", events_df.columns)

# TODO: create temp view = 5. Register a temporary view named `service_events`.
events_df.createOrReplaceTempView("service_events")

# TODO: run SQL checks
'''
6. Run SQL queries that answer:
   - How many rows are in the view?
   - Which services appear in the dataset?
   - How many rows does each region have?
'''
# How many rows are in the view?
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM service_events
""").show()

# - Which services appear in the dataset?
spark.sql("""
    SELECT DISTINCT service
    FROM service_events
""").show()

# - How many rows does each region have?
spark.sql("""
    SELECT region, COUNT(*) AS row_count
    FROM service_events
    GROUP BY region
""").show()

# 7. Stop Spark at the end.
spark.stop()

root
 |-- event_id: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- request_count: integer (nullable = true)
 |-- error_count: integer (nullable = true)
 |-- latency_ms: double (nullable = true)
 |-- bytes_in: double (nullable = true)
 |-- bytes_out: double (nullable = true)

+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|event_id|service        |region  |event_time         |request_count|error_count|latency_ms|bytes_in|bytes_out|
+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|1       |auth           |eu-west |2026-05-04 00:00:00|1240         |8          |82.4      |1.45E7  |3.82E7   |
|2       |payments       |eu-west |2026-05-04 00:00:00|860          |21         |146.2     |1.92E7  |4.41E7   |
|3       |search         |us-east |2026-05-04 01:00:00|211

### Session 9 | Part 2

> In Part 2, you will create derived columns and time features in Spark.

#### 1. Goal

You will:

- add calculated columns with `withColumn`
- use `when` for category labels
- calculate safe rates and traffic totals
- extract date, hour, and day of week from timestamps
- run SQL summaries using the new columns
- compare service-log features with final project market-data features

#### 2. Prerequisites

Before starting:

1. Finish [Part 1](./session-09-part-01.md).
2. Start Spark and load `service_events.csv` with the same schema.
3. Register the loaded DataFrame as `service_events`.

#### 3. Basics you should know

- `withColumn` adds or replaces a DataFrame column.
- `when(...).otherwise(...)` creates conditional values.
- `round` makes numeric output easier to read.
- `to_date`, `hour`, and `date_format` extract useful time features.
- Derived columns should be created before writing grouped analytics.

#### 4. Load the starting DataFrame

Use the same setup from Part 1. If you are continuing in the same notebook, you can reuse your existing `events_df`.

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

spark = (
    SparkSession.builder
    .appName("Session09Part02")
    .master("local[*]")
    .getOrCreate()
)

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

events_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv("service_events.csv")
)


#### 5. Add error and traffic columns

In [10]:
from pyspark.sql.functions import col, round, when

events_enriched_df = (
    events_df
    .withColumn(
        "error_rate",
        when(col("request_count") > 0, col("error_count") / col("request_count")).otherwise(0),
    )
    .withColumn("total_bytes", col("bytes_in") + col("bytes_out"))
    .withColumn("traffic_mb", col("total_bytes") / 1048576)
)

events_enriched_df.select(
    "service",
    "request_count",
    "error_count",
    round("error_rate", 4).alias("error_rate"),
    round("traffic_mb", 2).alias("traffic_mb"),
).show(8)

'''expected output:
+---------------+-------------+-----------+----------+----------+
|        service|request_count|error_count|error_rate|traffic_mb|
+---------------+-------------+-----------+----------+----------+
|           auth|         1240|          8|    0.0065|     50.26|
|       payments|          860|         21|    0.0244|     60.37|
|         search|         2110|         13|    0.0062|    100.14|
|recommendations|         1680|         18|    0.0107|    120.07|
|           auth|          980|          4|    0.0041|     39.77|
|       payments|          740|         17|     0.023|     58.27|
|         search|         2320|         15|    0.0065|    109.77|
|recommendations|         1760|         24|    0.0136|    128.75|
+---------------+-------------+-----------+----------+----------+
only showing top 8 rows
'''

+---------------+-------------+-----------+----------+----------+
|        service|request_count|error_count|error_rate|traffic_mb|
+---------------+-------------+-----------+----------+----------+
|           auth|         1240|          8|    0.0065|     50.26|
|       payments|          860|         21|    0.0244|     60.37|
|         search|         2110|         13|    0.0062|    100.14|
|recommendations|         1680|         18|    0.0107|    120.07|
|           auth|          980|          4|    0.0041|     39.77|
|       payments|          740|         17|     0.023|     58.27|
|         search|         2320|         15|    0.0065|    109.77|
|recommendations|         1760|         24|    0.0136|    128.75|
+---------------+-------------+-----------+----------+----------+
only showing top 8 rows


'expected output:\n+---------------+-------------+-----------+----------+----------+\n|        service|request_count|error_count|error_rate|traffic_mb|\n+---------------+-------------+-----------+----------+----------+\n|           auth|         1240|          8|    0.0065|     50.26|\n|       payments|          860|         21|    0.0244|     60.37|\n|         search|         2110|         13|    0.0062|    100.14|\n|recommendations|         1680|         18|    0.0107|    120.07|\n|           auth|          980|          4|    0.0041|     39.77|\n|       payments|          740|         17|     0.023|     58.27|\n|         search|         2320|         15|    0.0065|    109.77|\n|recommendations|         1760|         24|    0.0136|    128.75|\n+---------------+-------------+-----------+----------+----------+\nonly showing top 8 rows\n'

Checkpoint question:

Why do we check `request_count > 0` before calculating `error_rate`?

<details>
<summary>Show answer</summary>

It avoids dividing by zero. Even if this dataset has positive request counts, this is a safer habit for real analytics work.

</details>

#### 6. Add a latency band

In [11]:
events_enriched_df = events_enriched_df.withColumn(
    "latency_band",
    when(col("latency_ms") < 100, "fast")
    .when(col("latency_ms") < 160, "normal")
    .otherwise("slow"),
)

events_enriched_df.select("service", "latency_ms", "latency_band").show(8)
'''expected output:
+---------------+----------+------------+
|        service|latency_ms|latency_band|
+---------------+----------+------------+
|           auth|      82.4|        fast|
|       payments|     146.2|      normal|
|         search|      94.8|        fast|
|recommendations|     132.5|      normal|
|           auth|      78.1|        fast|
|       payments|     171.3|        slow|
|         search|     101.7|      normal|
|recommendations|     155.9|      normal|
+---------------+----------+------------+
only showing top 8 rows
'''

+---------------+----------+------------+
|        service|latency_ms|latency_band|
+---------------+----------+------------+
|           auth|      82.4|        fast|
|       payments|     146.2|      normal|
|         search|      94.8|        fast|
|recommendations|     132.5|      normal|
|           auth|      78.1|        fast|
|       payments|     171.3|        slow|
|         search|     101.7|      normal|
|recommendations|     155.9|      normal|
+---------------+----------+------------+
only showing top 8 rows


'expected output:\n+---------------+----------+------------+\n|        service|latency_ms|latency_band|\n+---------------+----------+------------+\n|           auth|      82.4|        fast|\n|       payments|     146.2|      normal|\n|         search|      94.8|        fast|\n|recommendations|     132.5|      normal|\n|           auth|      78.1|        fast|\n|       payments|     171.3|        slow|\n|         search|     101.7|      normal|\n|recommendations|     155.9|      normal|\n+---------------+----------+------------+\nonly showing top 8 rows\n'

Checkpoint question:

Which final project column is similar to this kind of category label?

<details>
<summary>Show answer</summary>

`candle_direction` is similar. It labels each row based on a condition: up, down, or flat.

</details>

#### 7. Add time features

In [12]:
from pyspark.sql.functions import date_format, hour, to_date

events_enriched_df = (
    events_enriched_df
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("day_of_week", date_format(col("event_time"), "E"))
)

events_enriched_df.select(
    "event_time",
    "event_date",
    "event_hour",
    "day_of_week",
).show(8, truncate=False)

+-------------------+----------+----------+-----------+
|event_time         |event_date|event_hour|day_of_week|
+-------------------+----------+----------+-----------+
|2026-05-04 00:00:00|2026-05-04|0         |Mon        |
|2026-05-04 00:00:00|2026-05-04|0         |Mon        |
|2026-05-04 01:00:00|2026-05-04|1         |Mon        |
|2026-05-04 01:00:00|2026-05-04|1         |Mon        |
|2026-05-04 02:00:00|2026-05-04|2         |Mon        |
|2026-05-04 02:00:00|2026-05-04|2         |Mon        |
|2026-05-04 03:00:00|2026-05-04|3         |Mon        |
|2026-05-04 03:00:00|2026-05-04|3         |Mon        |
+-------------------+----------+----------+-----------+
only showing top 8 rows


#### 8. Register the enriched view

In [13]:
events_enriched_df.createOrReplaceTempView("service_events_enriched")

Then query the new columns:

In [14]:
spark.sql("""
    SELECT
        service,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY service
    ORDER BY average_error_rate DESC
""").show()

+---------------+------------------+------------------+----------------+
|        service|average_error_rate|average_latency_ms|total_traffic_mb|
+---------------+------------------+------------------+----------------+
|       payments|            0.0243|            174.63|          407.89|
|recommendations|            0.0119|            140.75|          723.65|
|         search|            0.0057|             94.02|          614.26|
|           auth|            0.0051|             79.63|          305.56|
+---------------+------------------+------------------+----------------+



#### 9. Time-based SQL summary

In [15]:
spark.sql("""
    SELECT
        event_hour,
        SUM(request_count) AS total_requests,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY event_hour
    ORDER BY total_requests DESC
""").show()

+----------+--------------+----------------+
|event_hour|total_requests|total_traffic_mb|
+----------+--------------+----------------+
|         3|          8310|          484.18|
|         1|          7700|          447.94|
|        10|          6950|          405.79|
|         9|          5340|          285.82|
|         0|          4320|          226.69|
|         2|          3510|          200.94|
+----------+--------------+----------------+



Checkpoint question:

Why is `event_hour` useful for service analytics?

<details>
<summary>Show answer</summary>

It lets us compare activity by hour and identify when the system is busiest.

</details>

#### 10. Exercise 2: Derived columns and time analysis

## Exercise 2

Your notebook should:

1. Load the service events CSV.
2. Add `error_rate`, `total_bytes`, `traffic_mb`, and `latency_band`.
3. Add `event_date`, `event_hour`, and `day_of_week`.
4. Register a temporary view named `service_events_enriched`.
5. Write SQL queries that answer:
   - Which service has the highest average error rate?
   - Which service has the highest average latency?
   - Which hour has the most requests?
   - Which region has the most traffic?
6. Show a preview proving the derived columns exist.
7. Stop Spark at the end.


In [26]:
# Load the service events CSV.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Session09Exercise02")
    .master("local[*]")
    .getOrCreate()
)

events_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv("service_events.csv")
)

# Add error_rate, total_bytes, traffic_mb, and latency_band.
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

# Add event_date, event_hour, and day_of_week.
from pyspark.sql.functions import col, round, when, to_date, hour, date_format

events_enriched_df = (
    events_df
    .withColumn(
        "error_rate",
        when(col("request_count") > 0, col("error_count") / col("request_count")).otherwise(0),
    )
    .withColumn("total_bytes", col("bytes_in") + col("bytes_out"))
    .withColumn("traffic_mb", col("total_bytes") / 1048576)
    .withColumn(
        "latency_band",
        when(col("latency_ms") < 100, "fast")
        .when(col("latency_ms") < 160, "normal")
        .otherwise("slow"),)
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("day_of_week", date_format(col("event_time"), "E")))


# Register a temporary view named service_events_enriched.
events_enriched_df.createOrReplaceTempView("service_events_enriched")

# Write SQL queries that answer:
# Which service has the highest average error rate?


# Which service has the highest average latency?
spark.sql("""
    SELECT
        service,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY service
    ORDER BY average_error_rate DESC
""").show()

# Which hour has the most requests?
spark.sql("""
    SELECT
        event_hour,
        SUM(request_count) AS total_requests,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY event_hour
    ORDER BY total_requests DESC
""").show()

# Which region has the most traffic?
spark.sql("""
    SELECT
        region,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY region
    ORDER BY total_traffic_mb DESC
""").show()

# Show a preview proving the derived columns exist.
spark.sql("""
    SELECT *
    FROM service_events_enriched
    LIMIT 5
""").show()

# Stop Spark at the end.
spark.stop()

+---------------+------------------+------------------+----------------+
|        service|average_error_rate|average_latency_ms|total_traffic_mb|
+---------------+------------------+------------------+----------------+
|       payments|            0.0243|            174.63|          407.89|
|recommendations|            0.0119|            140.75|          723.65|
|         search|            0.0057|             94.02|          614.26|
|           auth|            0.0051|             79.63|          305.56|
+---------------+------------------+------------------+----------------+

+----------+--------------+----------------+
|event_hour|total_requests|total_traffic_mb|
+----------+--------------+----------------+
|         3|          8310|          484.18|
|         1|          7700|          447.94|
|        10|          6950|          405.79|
|         9|          5340|          285.82|
|         0|          4320|          226.69|
|         2|          3510|          200.94|
+---------